# The Vitamin Supplements Problem

In [24]:
import pandas as pd
import pulp as pp

In [25]:
tablets = pd.read_csv(f"https://github.com/shrutioak06/Shruti_DACSS690C/raw/main/Tablets%20Data.csv")
costs = pd.read_csv(f"https://github.com/shrutioak06/Shruti_DACSS690C/raw/main/Costs.csv")
tablets

,tablet,SuperVit,NewHealth,Condition
0,VitaminC,20,30,60
1,Calcium,500,250,1000
2,Iron,9,2,18
3,Niacin,2,10,20
4,Magnesium,60,90,360


In [26]:
costs

,product,costPerTablet
0,SuperVit,0.2
1,NewHealth,0.3


All of the steps below come directly from the lecture code, and are used as a reference point throughout.

# 1. Initialize the MODEL:

In [27]:
model = pp.LpProblem(name = "vitaminSupplements", sense = pp.LpMinimize)

# 2. Declare the VARIABLES:

In [28]:
SuperVit = pp.LpVariable(name = "SuperVit", lowBound = 0, cat = "Integer")
NewHealth = pp.LpVariable(name = "NewHealth", lowBound = 0, cat = "Integer")

# 3. Create function to OPTIMIZE:

In [29]:
costsSuperVit = float(costs.loc[costs["product"] == "SuperVit", "costPerTablet"].iloc[0])
costsNewHealth = float(costs.loc[costs["product"] == "NewHealth", "costPerTablet"].iloc[0])

obj_func = costsSuperVit * SuperVit + costsNewHealth * NewHealth

# 4. Represent the constraints:

In [30]:
constraints = {}
for _, row in tablets.iterrows():
    constraints[row["tablet"]] = pp.LpConstraint(
        name=row["tablet"],
        e = row["SuperVit"]*SuperVit + row["NewHealth"]*NewHealth,
        rhs=row["Condition"],
        sense=pp.LpConstraintGE) # >= 'sense'

C1 = constraints["VitaminC"]
C2 = constraints["Calcium"]
C3 = constraints["Iron"]
C4 = constraints["Niacin"]
C5 = constraints["Magnesium"]

# 5. Build MODEL:

In [31]:
model += obj_func
model += C1
model += C2
model += C3
model += C4
model += C5

# 6. Solve the MODEL:

In [32]:
model.solve(pp.PULP_CBC_CMD(msg=0))

1

# 7. Basic Report

In [33]:
"Model Status",pp.LpStatus[model.status]

('Model Status', 'Optimal')

In [34]:
Optimal_Decision={"Optimal Solution to minimize cost":pp.value(model.objective)}
Optimal_Decision.update({v.name: v.varValue for v in model.variables()})
Optimal_Decision

{'Optimal Solution to minimize cost': 1.2000000000000002,
 'NewHealth': 2.0,
 'SuperVit': 3.0}

In [35]:
pd.DataFrame.from_dict(Optimal_Decision,orient='index',columns=['info']).map('{:,.2f}'.format)

,info
Optimal Solution to minimize cost,1.20
NewHealth,2.00
SuperVit,3.00


Let's read the output row by row:

1. The first row answers the question we asked the model:

Minimium daily cost = $1.20

2. The second row shows how many NewHealth tablets to buy: 2
3. The third row shows how many SuperVit tablets to buy: 3

This is the cheapest combination that still satisfies all five minimum conditions.

# 8. The Sensitivity of the Result

Since tablets cannot be a fractional value, the variables are declared as Integer. c.pi will return either None or 0.0 for every constraint because integer decisions jump.

In [36]:
import numpy as np

tolerance = 1e-6

sense_names = {-1: "<=", 0: "=", 1: ">="}

sensitivityDF = (
    pd.DataFrame([
        {
            "constraint": name,
            "sense": sense_names[c.sense],
            "shadow_price (dual_value)": c.pi,
            "distance_to_limit (slack/surplus)": abs(c.slack)
        }
        for name, c in model.constraints.items()
    ])
    .assign(
        binding=lambda df: np.where(
            df["distance_to_limit (slack/surplus)"] <= tolerance, "Yes", "No"
        )
    )
    .set_index("constraint")
)

sensitivityDF

,sense,shadow_price (dual_value),distance_to_limit (slack/surplus),binding
constraint,,,,
VitaminC,>=,0.0,60.0,No
Calcium,>=,0.0,1000.0,No
Iron,>=,0.0,13.0,No
Niacin,>=,0.0,6.0,No
Magnesium,>=,0.0,0.0,Yes


As expected, the shadow_price (dual_value) is 0.0 for every constraint, although the binding shows exactly which constraints are hit. Which is only Magnesium in this case. These results are consistent with the expected outcomes demonstrated in the slides, confirming that the methodology is accurate.